In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

# Placeholders à remplacer avec les vraies fonctions
def simulatePendulumCart(params, theta_des, simu_time, dt):
    # Remplacer ceci par le vrai modèle
    settling_time = np.random.uniform(0, 5)
    static_error = np.random.uniform(0, 1)
    return settling_time, static_error

def fitnesse(settling_time, static_error, flag):
    if flag == 0:
        return settling_time + 10 * static_error
    else:
        return 10 * static_error

def final_simu(description, fitness, pop):
    print(f"Final simulation ({description}): Best fitness = {min(fitness)}")
    return pop[np.argmin(fitness)]


In [ ]:
popsize = 50
maxgen = 50
simu_time = 10  # secondes
elitism_rate = 0.1
selection_rate = 0.5
theta_des = 0
dt = 0.01

pop = []
fitness = []
flag = 0

for _ in range(popsize):
    chr = np.zeros((15, 3))
    chr[0, 0] = -np.pi
    chr[2, 2] = np.pi
    chr[3, 0] = -10
    chr[5, 2] = 10

    angles = np.sort(2 * np.pi * np.random.rand(7) - np.pi)
    speeds = np.sort(20 * np.random.rand(7) - 10)
    params = np.vstack((np.concatenate(([-np.pi], angles, [np.pi])),
                        np.concatenate(([-10], speeds, [10]))))

    chr[0:3, :] = params[0, [0, 1, 3]], params[0, [2, 4, 6]], params[0, [5, 7, 8]]
    chr[3:6, :] = params[1, [0, 1, 3]], params[1, [2, 4, 6]], params[1, [5, 7, 8]]

    for k in range(6, 15):
        chr[k, :] = 20 * np.random.rand(3) - 10

    pop.append(chr)


In [ ]:
fitness = []
for chr in pop:
    settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
    fitness.append(fitnesse(settling_time, static_error, flag))


In [ ]:
def selection(pop, fitness_vals, rate, popsize, elitism_rate):
    num_selected = round(rate * popsize)
    num_elites = int(np.ceil(elitism_rate * popsize))
    sorted_idx = np.argsort(fitness_vals)
    elites = [pop[i] for i in sorted_idx[:num_elites]]

    selected = []
    for _ in range(num_selected - num_elites):
        candidates = random.sample(range(popsize), 3)
        best = min(candidates, key=lambda idx: fitness_vals[idx])
        selected.append(pop[best])

    return num_elites, elites + selected

def mutation(child, mutation_rate, startMF_rad, endMF_rad, startMF_speed, endMF_speed):
    if random.random() < mutation_rate:
        for i in range(3):
            for n in range(15):
                multiplier = 1 + 0.6 * (random.random() - 0.5)
                child[n, i] *= multiplier

        child[0, 0] = startMF_rad
        child[2, 2] = endMF_rad
        child[3, 0] = startMF_speed
        child[5, 2] = endMF_speed
    return child

def reproduction(pool, mutation_rate, popsize, num_elites):
    new_pop = []
    for i in range(num_elites):
        new_pop.append(pool[i])
    for i in range(num_elites, popsize):
        parent1 = random.choice(pool)
        parent2 = random.choice(pool)
        child = np.zeros((15, 3))
        for k in range(15):
            alpha = random.random()
            beta = 1 - alpha
            child[k, :] = alpha * parent1[k, :] + beta * parent2[k, :]
        child = mutation(child, mutation_rate, -np.pi, np.pi, -10, 10)
        new_pop.append(child)
    return new_pop


In [ ]:
evo = []
gen = 0
while gen < maxgen:
    mutation_rate = max(0.3, 1 - gen / maxgen)
    num_elites, pool = selection(pop, fitness, selection_rate, popsize, elitism_rate)
    pop = reproduction(pool, mutation_rate, popsize, num_elites)
    fitness = []
    for chr in pop:
        settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
        fitness.append(fitnesse(settling_time, static_error, flag))
    evo.append(min(fitness))
    print(f"Generation {gen}, Best Fitness: {evo[-1]}")
    gen += 1
out1 = final_simu("with settling time", fitness, pop)


In [ ]:
# Nouvelle initialisation
flag = 1
pop = []
fitness = []
for _ in range(popsize):
    chr = np.zeros((15, 3))
    chr[0, 0] = -np.pi
    chr[2, 2] = np.pi
    chr[3, 0] = -10
    chr[5, 2] = 10

    angles = np.sort(2 * np.pi * np.random.rand(7) - np.pi)
    speeds = np.sort(20 * np.random.rand(7) - 10)
    params = np.vstack((np.concatenate(([-np.pi], angles, [np.pi])),
                        np.concatenate(([-10], speeds, [10]))))

    chr[0:3, :] = params[0, [0, 1, 3]], params[0, [2, 4, 6]], params[0, [5, 7, 8]]
    chr[3:6, :] = params[1, [0, 1, 3]], params[1, [2, 4, 6]], params[1, [5, 7, 8]]

    for k in range(6, 15):
        chr[k, :] = 20 * np.random.rand(3) - 10

    pop.append(chr)

for chr in pop:
    settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
    fitness.append(fitnesse(settling_time, static_error, flag))

evo = []
gen = 0
while gen < maxgen:
    mutation_rate = max(0.1, 1 - gen / maxgen)
    num_elites, pool = selection(pop, fitness, selection_rate, popsize, elitism_rate)
    pop = reproduction(pool, mutation_rate, popsize, num_elites)
    fitness = []
    for chr in pop:
        settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
        fitness.append(fitnesse(settling_time, static_error, flag))
    evo.append(min(fitness))
    print(f"Generation {gen}, Best Fitness: {evo[-1]}")
    gen += 1
out2 = final_simu("without settling time", fitness, pop)
